# WM Guidance Extractor — Fine-tune LLaMA 3.1

**Runtime**: GPU → T4

Steps:
1. Install Unsloth
2. Upload training / validation data
3. Load base model + apply QLoRA
4. Train
5. Evaluate on validation set
6. Export to GGUF for Ollama

## 1. Install

In [1]:
!pip install unsloth
!pip install --upgrade transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

## 2. Upload data files

In [21]:
from google.colab import files

print('Upload guidance_train.jsonl, guidance_val.jsonl and guidance_test.jsonl')
uploaded = files.upload()  # select both files from data/training/

Upload guidance_train.jsonl, guidance_val.jsonl and guidance_test.jsonl


Saving guidance_test.jsonl to guidance_test.jsonl
Saving guidance_train.jsonl to guidance_train (1).jsonl
Saving guidance_val.jsonl to guidance_val (1).jsonl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Load base model + QLoRA

In [11]:
from unsloth import FastLanguageModel
import torch
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MAX_SEQ_LEN = 1024

# LLaMA 3.2 3B — fits T4 comfortably; upgrade to 8B on Colab Pro A100
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(model.print_trainable_parameters())

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
None


## 4. Prepare dataset

In [22]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

train_ds = load_dataset("json", data_files="guidance_train.jsonl", split="train")
val_ds   = load_dataset("json", data_files="guidance_val.jsonl",   split="train")

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

def format_example(example):
    """Convert ShareGPT conversations → tokenized chat string."""
    messages = [
        {"role": "system" if c["from"] == "system" else
                 "user"   if c["from"] == "human"  else "assistant",
         "content": c["value"]}
        for c in example["conversations"]
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
val_ds   = val_ds.map(format_example)

print(train_ds[0]["text"][:400])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 1600  Val: 250


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Extract only explicitly stated financial guidance values from a WM earnings press release. Return valid JSON only. Use null for missing fields. Do not guess.
Rules:
- revenue_min/revenue_max: Total Company revenue guidance only. Ignore segment, collection, disposal, internal


## 5. Train

In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8, 
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=False, 
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="wm_guidance_lora",
        report_to="none",
        dataloader_pin_memory=False,
    ),
)

trainer_stats = trainer.train()
print(f"\nDone. Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1600 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/250 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,600 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Epoch,Training Loss,Validation Loss
1,0.104795,0.107201
2,0.103103,0.104579
3,0.102184,0.103800


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


Done. Runtime: 3170s


## 6. Evaluate on validation set

In [24]:
import json
import torch

FastLanguageModel.for_inference(model)

# Fix: Unsloth sets _per_layer_device_index=None for some layers on single-GPU setups
base = model.base_model.model if hasattr(model, "base_model") else model
for layer in base.model.layers:
    if getattr(layer, "_per_layer_device_index", None) is None:
        layer._per_layer_device_index = 0

val_raw = [json.loads(l) for l in open("guidance_test.jsonl")]

for ex in val_raw:
    convs = ex["conversations"]
    messages = [
        {"role": "system" if c["from"] == "system" else "user",
         "content": c["value"]}
        for c in convs if c["from"] != "gpt"
    ]
    expected = next(c["value"] for c in convs if c["from"] == "gpt")

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    try:
        p = json.loads(pred)
        e = json.loads(expected)
        match = all(p.get(k) == e.get(k) for k in e)
        status = "✓" if match else "✗"
    except Exception:
        status = "ERR"

    section = convs[1]["value"].split("\n")[0]
    print(f"{status}  {section[:50]}")
    if status != "✓":
        print(f"   expected: {expected}")
        print(f"   got:      {pred}")

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

✓  Section title: OUTLOOK


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓  Section title: REVISED 2021 OUTLOOK


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

✓  Section title: 2025 FINANCIAL OUTLOOK


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓  Section title: 2025 Outlook


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓  Section title: 2025 Outlook
✗  Section title: FINANCIAL OUTLOOK
   expected: {"revenue_min":26425,"revenue_max":26625,"revenue_unit":"million","fcf_min":3750,"fcf_max":3850,"fcf_unit":"million"}
   got:      {"revenue_min":26425,"revenue_max":26625,"revenue_unit":"million","fcf_min":3750,"fcf_max":4850,"fcf_unit":"million"}


## 7. Export to GGUF (for Ollama)

In [25]:
# Save merged model as GGUF Q4_K_M — ~4.5 GB, good balance of speed/quality
model.save_pretrained_gguf(
    "wm_guidance_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved to wm_guidance_gguf/")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [09:14<00:00, 277.03s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:17<00:00, 98.76s/it]


Unsloth: Merge process complete. Saved to `/content/wm_guidance_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['wm_guidance_gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['wm_guidance_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model wm_guidance_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to wm_guidance_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f wm_guidance_gguf_gguf/Modelfile
Saved to wm_guidance_gguf/


In [27]:
# Unsloth appends _gguf to the output dir name → actual path is wm_guidance_gguf_gguf
import glob

gguf_files = glob.glob("wm_guidance_gguf_gguf/*.gguf")
modelfile   = glob.glob("wm_guidance_gguf_gguf/Modelfile")
print("GGUF files:", gguf_files)
print("Modelfile: ", modelfile)

# Download only the Q4_K_M quantized model + Modelfile (skip the large F16 file)
from google.colab import files
for f in gguf_files:
    if "Q4_K_M" in f:
        print(f"Downloading {f} ...")
        files.download(f)
for f in modelfile:
    files.download(f)

GGUF files: ['wm_guidance_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Modelfile:  ['wm_guidance_gguf_gguf/Modelfile']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Load into Ollama

```bash
# Create a Modelfile
cat > Modelfile <<'EOF'
FROM ./wm_guidance_gguf/unsloth.Q4_K_M.gguf
EOF

# Register model
ollama create wm-guidance -f Modelfile

# Test
ollama run wm-guidance
```

Then in `local_guidance_extractor.py`:
```python
extractor = LocalGuidanceExtractor(model="wm-guidance")
```